# 03 機率與離散分布（講義 04a / 04b / 04c、HW04）

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

## 排列組合

In [ ]:
print('C(10, 3) =', math.comb(10, 3))
print('P(10, 3) =', math.perm(10, 3))
print('5!       =', math.factorial(5))

## 機率法則、貝氏定理

- 加法：$\Pr(A \cup B) = \Pr(A) + \Pr(B) - \Pr(A \cap B)$
- 條件：$\Pr(A \mid B) = \frac{\Pr(A \cap B)}{\Pr(B)}$；獨立：$\Pr(A \cap B) = \Pr(A)\Pr(B)$
- 全機率：$\Pr(B) = \sum_i \Pr(B \mid A_i)\Pr(A_i)$
- 貝氏：$\Pr(A_i \mid B) = \frac{\Pr(B \mid A_i)\Pr(A_i)}{\sum_j \Pr(B \mid A_j)\Pr(A_j)}$

In [ ]:
prior = {'有病': 0.01, '沒病': 0.99}
likelihood = {'有病': 0.95, '沒病': 0.05}          # Pr(陽性 | 狀態)
evidence = sum(prior[a] * likelihood[a] for a in prior)
posterior = {a: prior[a] * likelihood[a] / evidence for a in prior}
print('Pr(陽性) =', round(evidence, 4))
print('Pr(有病 | 陽性) =', round(posterior['有病'], 4))

## 聯合機率表（講義 04a 洗衣店例題）

已知：51% 認為折扣重要（I）、57% 會因折扣換店（S）、認為折扣重要的人有 90% 會換店（$\Pr(S \mid I) = 0.9$）。

- 乘法：$\Pr(S \cap I) = \Pr(S \mid I)\Pr(I)$
- 每一列、每一欄加總 = 邊際機率
- 條件機率 = 聯合 ÷ 邊際：$\Pr(S \mid U) = \frac{\Pr(S \cap U)}{\Pr(U)}$
- 獨立 ⇔ 每一格都等於邊際相乘

In [ ]:
p_i, p_s, p_s_given_i = 0.51, 0.57, 0.9
p_s_and_i = p_s_given_i * p_i                # 乘法
p_n_and_i = p_i - p_s_and_i                  # 同一列加總 = 邊際
p_s_and_u = p_s - p_s_and_i                  # 同一欄加總 = 邊際
p_n_and_u = 1 - p_s_and_i - p_n_and_i - p_s_and_u
table = pd.DataFrame([[p_s_and_i, p_n_and_i], [p_s_and_u, p_n_and_u]],
                     index=['Important', 'Unimportant'], columns=['Switch', "Don't switch"])
table['Total'] = table.sum(axis=1)
table.loc['Total'] = table.sum()
display(table.round(3))
print('Pr(S | U) =', round(p_s_and_u / (1 - p_i), 4))        # 講義 ≈ 0.227
print('獨立嗎？', abs(p_s_and_i - p_i * p_s) < 1e-9)

In [ ]:
sk.joint([[0.459, 0.051], [0.111, 0.379]], xs=['Important', 'Unimportant'],
         ys=['Switch', "Don't switch"], conditional='x')

## 貝氏：參數的後驗分布

不確定參數（例如硬幣正面機率 p）時，先給每個可能值一個**先驗**機率；觀察到結果後，**後驗** = 先驗 × 概似 ÷ 總和。概似就是「在這個 p 下，觀察到這個結果的機率」，常用二項或幾何的 pmf。

In [ ]:
params = [0.3, 0.5, 0.7]
prior = [1/3, 1/3, 1/3]

likelihood = [stats.binom.pmf(2, 5, p) for p in params]       # 丟 5 次、出現 2 次正面
total = sum(a * b for a, b in zip(prior, likelihood))
print('二項：', [round(a * b / total, 4) for a, b in zip(prior, likelihood)])

likelihood = [stats.geom.pmf(3, p) for p in params]           # 一直丟到正面：第 3 次才出現
total = sum(a * b for a, b in zip(prior, likelihood))
print('幾何：', [round(a * b / total, 4) for a, b in zip(prior, likelihood)])

In [ ]:
sk.posterior({0.3: 1/3, 0.5: 1/3, 0.7: 1/3}, lambda p: stats.geom.pmf(3, p))

## 隨機變數的期望值與變異數（講義 04b）

$E[a + bX] = a + bE[X]$、$\operatorname{Var}(a + bX) = b^2\operatorname{Var}(X)$；
$E[\sum X_i] = \sum E[X_i]$；獨立時 $\operatorname{Var}(\sum X_i) = \sum \operatorname{Var}(X_i)$

In [ ]:
xs = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8])
ps = np.array([0.06, 0.15, 0.22, 0.22, 0.17, 0.10, 0.05, 0.02, 0.01])   # 講義 04b 蘋果需求
EX = (xs * ps).sum()
VarX = ((xs - EX) ** 2 * ps).sum()
print('E[X] =', EX, ' Var(X) =', round(VarX, 4), ' SD =', round(VarX ** 0.5, 4))
print('F(x) =', np.cumsum(ps).round(2))

D = sk.Discrete(xs, ps)    # 同樣的機率表，可以直接問機率
D.P('X <= 4')
D.quantile(0.8)            # 80% 服務水準的最小存貨

In [ ]:
sk.inventory_profit(D, price=10, cost=2)   # 講義 04b：每個進貨量的期望利潤

## 兩個隨機變數：共變異數、相關係數、不相關 vs 獨立

- $\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)] = E[XY] - \mu_X\mu_Y$，$\rho = \frac{\text{Cov}(X, Y)}{\sigma_X\sigma_Y}$
- **獨立 ⇒ 不相關（Cov = 0）**，反過來不成立
- $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\text{Cov}(X, Y)$：只要不相關，變異數就能直接相加

In [ ]:
xs, ys = [0, 1], [1, 2, 3]
p = [[0.1, 0.2, 0.2],
     [0.2, 0.2, 0.1]]                         # p[i][j] = Pr(X = xs[i], Y = ys[j])
px = [sum(row) for row in p]
py = [sum(p[i][j] for i in range(len(xs))) for j in range(len(ys))]
mu_x = sum(x * q for x, q in zip(xs, px))
mu_y = sum(y * q for y, q in zip(ys, py))
exy = sum(xs[i] * ys[j] * p[i][j] for i in range(len(xs)) for j in range(len(ys)))
cov = exy - mu_x * mu_y
var_x = sum(x ** 2 * q for x, q in zip(xs, px)) - mu_x ** 2
var_y = sum(y ** 2 * q for y, q in zip(ys, py)) - mu_y ** 2
print('Cov =', round(cov, 4), ' ρ =', round(cov / math.sqrt(var_x * var_y), 4))

**不相關但不獨立**的典型例子：$Y = X^2$，$X$ 是 −1、0、1。

In [ ]:
sk.joint([[0, 0.25], [0.5, 0], [0, 0.25]], xs=[-1, 0, 1], ys=[0, 1])

## 獨立的離散與連續隨機變數

- 獨立 → 聯合機率 = 各自的機率相乘
- 連續隨機變數的**單點機率是 0**，所以 $\Pr(X_1 = 2, X_2 = 2) = 0$
- $\Pr(X_1 + X_2 \ge c)$：對離散的 $X_1$ 每個值分情況（全機率），$\sum_k \Pr(X_1 = k)\Pr(X_2 \ge c - k)$

In [ ]:
X1 = stats.binom(4, 0.5)                      # 離散 Bi(4, 0.5)
X2 = stats.uniform(loc=0, scale=5)            # 連續 Uni(0, 5)
print('Pr(X1 ≥ 2, X2 ≥ 3) =', round((1 - X1.cdf(1)) * (1 - X2.cdf(3)), 4))      # 獨立 → 相乘
total = sum(X1.pmf(k) * (1 - X2.cdf(3 - k)) for k in range(0, 5))              # 分情況加總
print('Pr(X1 + X2 ≥ 3) =', round(total, 4))
print('Var(X1 + X2) =', X1.var() + X2.var())                                     # 獨立 → 變異數相加

## E[g(X)]：含獎金、罰款的期望收益

$E[g(X)] = \sum_x g(x)\Pr(X = x)$。條件寫成 `(x >= 10)` 會是 True = 1／False = 0，可以直接乘。也可以拆開：$3E[X] + 15\Pr(X \ge 10) - 5\Pr(X \le 3)$。

In [ ]:
lam = 6                                     # 每小時平均 6 位顧客
def g(x):                                   # 每位 3 元；至少 10 位加 15 元；不超過 3 位罰 5 元
    return 3 * x + 15 * (x >= 10) - 5 * (x <= 3)
print(round(sum(g(x) * stats.poisson.pmf(k=x, mu=lam) for x in range(0, 100)), 4))
print(round(3 * lam + 15 * (1 - stats.poisson.cdf(k=9, mu=lam)) - 5 * stats.poisson.cdf(k=3, mu=lam), 4))   # 拆開算

In [ ]:
sk.expect(sk.Poisson(6), g)

## 二項分布 Bi(n, p)

$\Pr(X = x) = \binom{n}{x} p^x (1-p)^{n-x}$，$E[X] = np$，$\operatorname{Var}(X) = np(1-p)$

In [ ]:
trial_cnt, success_prob = 10, 0.4
print('Pr(X = 5)  =', round(stats.binom.pmf(k=5, n=trial_cnt, p=success_prob), 4))
print('Pr(X <= 4) =', round(stats.binom.cdf(k=4, n=trial_cnt, p=success_prob), 4))
print('Pr(X >= 3) =', round(1 - stats.binom.cdf(k=2, n=trial_cnt, p=success_prob), 4))   # ≥ 3 → 1 - F(2)
print('min x with F(x) >= 0.8:', int(stats.binom.ppf(q=0.8, n=trial_cnt, p=success_prob)))
print('mean, var:', stats.binom.mean(n=trial_cnt, p=success_prob), stats.binom.var(n=trial_cnt, p=success_prob))

用 statkit 驗算（**會自動處理 ≥ / > 要不要減 1**，並印出可以貼上的程式碼）：

In [ ]:
X = sk.Binom(10, 0.4)
X.P('X >= 3')
X.P('2 < X <= 5')
X.moments()
sk.plot_dist(X, 'X >= 3')
plt.show()

## 獨立二項的和（講義 04c）

$X \sim \text{Bi}(n_1, p)$、$Y \sim \text{Bi}(n_2, p)$ 獨立且 **p 相同** → $X + Y \sim \text{Bi}(n_1 + n_2, p)$。p 不同就不是二項。

In [ ]:
rng = np.random.default_rng(1)
total = stats.binom.rvs(n=6, p=0.3, size=100000, random_state=rng) + stats.binom.rvs(n=4, p=0.3, size=100000, random_state=rng)
print('模擬 Pr(X + Y = 3) ≈', round((total == 3).mean(), 4))
print('Bi(10, 0.3)     =', round(stats.binom.pmf(k=3, n=10, p=0.3), 4))

## 卜瓦松分布 Poi(λ)

$\Pr(X = x) = \frac{e^{-\lambda}\lambda^x}{x!}$，$E[X] = \operatorname{Var}(X) = \lambda$。λ 要跟題目的時間單位一致（例：每小時 3 次 → 20 分鐘 λ = 1）。

In [ ]:
print('Pr(X >= 3) =', round(1 - stats.poisson.cdf(k=2, mu=3), 4))
print('Pr(X = 7)  =', round(stats.poisson.pmf(k=7, mu=3), 4))
sk.Poisson(6.6).P('X >= 10')

## 超幾何分布 Hyper(N, K, n)：不放回抽樣

⚠ scipy 參數名稱：`M` = 母體 N、`n` = 成功數 K、`N` = 抽出數 n

In [ ]:
print('Pr(X = 7)  =', round(stats.hypergeom.pmf(k=7, M=20, n=12, N=10), 4))
print('Pr(X <= 1) =', round(stats.hypergeom.cdf(k=1, M=5, n=2, N=3), 4))
sk.Hypergeom(N=20, K=12, n=10).P('X >= 8')

## 超幾何 ≈ 二項（講義 04c）

不放回抽樣時，抽的比例 $\frac{n}{N}$ 很小 → 每次抽到成功的機率幾乎不變，可以用 $\text{Bi}\left(n, \frac{K}{N}\right)$ 近似。超幾何的變異數多了 $\frac{N - n}{N - 1}$，比二項小（更集中）。

In [ ]:
for N in [20, 200, 2000]:
    K = int(N * 0.3)
    print(f'N = {N:4d}：Hyper = {stats.hypergeom.pmf(k=3, M=N, n=K, N=10):.4f}，Bi(10, 0.3) = {stats.binom.pmf(k=3, n=10, p=0.3):.4f}')

## 幾何分布 Geo(p)：第一次成功在第 X 次（X = 1, 2, ...）

$\Pr(X = x) = (1-p)^{x-1}p$，$E[X] = 1/p$，$\operatorname{Var}(X) = (1-p)/p^2$

In [ ]:
print('Pr(X = 5)  =', round(stats.geom.pmf(k=5, p=0.4), 4))
print('Pr(X <= 4) =', round(stats.geom.cdf(k=4, p=0.4), 4))
sk.Geom(0.4).moments()

## 二項的卜瓦松近似（n 大、p 小，λ = np）

In [ ]:
n, p = 100, 0.03
print('binomial:', round(stats.binom.cdf(k=2, n=n, p=p), 4))
print('poisson :', round(stats.poisson.cdf(k=2, mu=n * p), 4))

## PMF 長條圖（課程寫法）

In [ ]:
possible_values = list(range(0, 21))
probabilities = stats.poisson.pmf(k=possible_values, mu=4)
plt.bar(possible_values, probabilities, facecolor='lightgray', edgecolor='black')
plt.xlabel('Number of occurrences')
plt.ylabel('Probability')
plt.title('The Poisson Distribution (mu = 4)')
plt.show()

## 用模擬驗證期望值與變異數

In [ ]:
random_samples = stats.binom.rvs(n=10, p=0.4, size=100000, random_state=115)
print('sample mean / var:', round(random_samples.mean(), 4), round(random_samples.var(ddof=1), 4))
print('theory      :', 10 * 0.4, 10 * 0.4 * 0.6)